### Final Team Project: Music Genre and Composer Classification Using Deep Learning
##### Name: Carlo Casella, Jasmine Duong, Emmanuel Sadek

##### Introduction
Music is a form of art that is ubiquitous and has a rich history. Different composers have created music with their unique styles and compositions. However, identifying the composer of a particular piece of music can be a challenging task, especially for novice musicians or listeners. The proposed project aims to use deep learning techniques to identify the composer of a given piece of music accurately.

##### Objective
The primary objective of this project is to develop a deep learning model that can predict the composer of a given musical score accurately. The project aims to accomplish this objective by using two deep learning techniques: Long Short-Term Memory (LSTM) and Convolutional Neural Network (CNN).

Dataset
The project will use a dataset consisting of musical scores from various composers. 

The dataset contains the midi files of compositions from well-known classical composers like Bach, Beethoven, Chopin, and Mozart. The dataset should be labeled with the name of the composer for each score. Please only do your prediction only for below composers, therefore you need to select the required composers from the given dataset above.

1-Bach
2-Beethoven
3-Chopin
4-Mozart

In [0]:
%%capture
!pip install --upgrade 'protobuf>=6,<7'
dbutils.library.restartPython()

In [0]:
%%capture

!pip install pretty_midi music21 torch torchvision torchaudio scikit-learn kagglehub tensorflow

import os
import shutil
import glob
import random
import numpy as np

import pretty_midi
from music21 import converter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix


#### 1. Dataset filtering (only Bach, Beethoven, Chopin, Mozart)

In [0]:
import kagglehub
path = kagglehub.dataset_download("blanderbuss/midi-classic-music")
print("Path to dataset files:", path)


In [0]:
# Clone github repo to grab supplemental data
# repo_url = "https://github.com/jazz1416/Composer-Identification-using-Neural-Networks"
# os.system(f"git clone {repo_url}")

# git_beethoven_dir = os.path.join("Composer-Identification-using-Neural-Networks", "beeth")
# git_mozart_dir = os.path.join("Composer-Identification-using-Neural-Networks", "mozart")
# git_chopin_dir = os.path.join("Composer-Identification-using-Neural-Networks", "chopin")

In [0]:
SOURCE_DIR = path  # from kagglehub
FILTERED_DIR = "filtered_composers"
os.makedirs(FILTERED_DIR, exist_ok=True)

TARGET_COMPOSERS = ["Bach", "Beethoven", "Chopin", "Mozart"]

for comp in TARGET_COMPOSERS:
    os.makedirs(os.path.join(FILTERED_DIR, comp), exist_ok=True)

for root, dirs, files in os.walk(SOURCE_DIR):
    for file in files:
        if file.lower().endswith((".mid", ".midi")):
            full_path = os.path.join(root, file)
            lower_root = root.lower()
            lower_file = file.lower()
            for comp in TARGET_COMPOSERS:
                if comp.lower() in lower_root or comp.lower() in lower_file:
                    dest_dir = os.path.join(FILTERED_DIR, comp)
                    shutil.copy(full_path, dest_dir)
                    break

In [0]:
import os
import random

# Limit the Bach MIDI files
FILTERED_DIR = "filtered_composers"
BACH_DIR = os.path.join(FILTERED_DIR, "Bach")
BACH_LIMIT = 250

if os.path.exists(BACH_DIR):
    # List all files currently in the Bach folder
    bach_files = [f for f in os.listdir(BACH_DIR) if os.path.isfile(os.path.join(BACH_DIR, f))]
    
    if len(bach_files) > BACH_LIMIT:
        print(f"Current Bach count: {len(bach_files)}. Pruning to {BACH_LIMIT}...")
        
        # Randomly select the files that will be deleted
        files_to_remove = random.sample(bach_files, len(bach_files) - BACH_LIMIT)
        
        for f in files_to_remove:
            os.remove(os.path.join(BACH_DIR, f))
            
        # Verify the new count
        new_count = len(os.listdir(BACH_DIR))
        print(f"Successfully removed {len(files_to_remove)} files. New Bach count: {new_count}")
    else:
        print(f"Bach directory already has {len(bach_files)} files. No action taken.")
else:
    print("Bach directory not found within FILTERED_DIR.")

In [0]:
# import os
# import shutil
# import hashlib

# # Define directories 
# SOURCE_DIR = path  # Kaggle path
# BEETH_SOURCE_DIR = git_beethoven_dir 
# MOZART_SOURCE_DIR = git_mozart_dir
# CHOPIN_SOURCE_DIR = git_chopin_dir
# FILTERED_DIR = "filtered_composers"
# TARGET_COMPOSERS = ["Bach", "Beethoven", "Chopin", "Mozart"]

# # Setup hashes to eliminate duplicates
# seen_hashes = set()

# def get_file_hash(file_path):
#     """Creates a unique signature for binary MIDI content."""
#     hasher = hashlib.md5()
#     with open(file_path, 'rb') as f:
#         buf = f.read()
#         hasher.update(buf)
#     return hasher.hexdigest()

# # Create destination folder
# for comp in TARGET_COMPOSERS:
#     os.makedirs(os.path.join(FILTERED_DIR, comp), exist_ok=True)

# # Process al directories
# all_sources = [SOURCE_DIR, BEETH_SOURCE_DIR, MOZART_SOURCE_DIR, CHOPIN_SOURCE_DIR]

# for source in all_sources:
#     if not os.path.exists(source):
#         continue
        
#     for root, dirs, files in os.walk(source):
#         for file in files:
#             if file.lower().endswith((".mid", ".midi")):
#                 full_path = os.path.join(root, file)
                
#                 # Check for duplicates across sources
#                 file_hash = get_file_hash(full_path)
#                 # If file content already processed, skip
#                 if file_hash in seen_hashes:
#                     continue 
                
#                 # Filter by composer name
#                 lower_root = root.lower()
#                 lower_file = file.lower()
#                 for comp in TARGET_COMPOSERS:
#                     if comp.lower() in lower_root or comp.lower() in lower_file:
#                         dest_dir = os.path.join(FILTERED_DIR, comp)
#                         shutil.copy(full_path, dest_dir)
#                         seen_hashes.add(file_hash)
#                         break

#### 2. Exploratory Data Analysis

In [0]:
import os
import pretty_midi
import numpy as np
import matplotlib.pyplot as plt

composer_counts = {c: len(os.listdir(os.path.join(FILTERED_DIR, c))) for c in TARGET_COMPOSERS}
composer_counts


##### 2.1 Count files per composer

In [0]:
plt.bar(composer_counts.keys(), composer_counts.values())
plt.title("Number of MIDI Files per Composer")
plt.ylabel("Count")
plt.show()

##### 2.2 Extract Global Statistics - Pitch distribution

In [0]:
all_pitches = []

for comp in TARGET_COMPOSERS:
    comp_dir = os.path.join(FILTERED_DIR, comp)
    for f in os.listdir(comp_dir):
        try:
            pm = pretty_midi.PrettyMIDI(os.path.join(comp_dir, f))
            for inst in pm.instruments:
                for note in inst.notes:
                    all_pitches.append(note.pitch)
        except:
            continue

plt.hist(all_pitches, bins=60)
plt.title("Global Pitch Distribution")
plt.xlabel("MIDI Pitch")
plt.ylabel("Frequency")
plt.show()

##### 2.3 Extract Global Statistics - Duration distribution

In [0]:
durations = []

for comp in TARGET_COMPOSERS:
    comp_dir = os.path.join(FILTERED_DIR, comp)
    for f in os.listdir(comp_dir):
        try:
            pm = pretty_midi.PrettyMIDI(os.path.join(comp_dir, f))
            for inst in pm.instruments:
                for note in inst.notes:
                    durations.append(note.end - note.start)
        except:
            continue

plt.hist(durations, bins=50)
plt.title("Note Duration Distribution")
plt.xlabel("Seconds")
plt.ylabel("Frequency")
plt.show()

##### 2.4 Extract Global Statistics - Tempo distribution

In [0]:
tempos = []

for comp in TARGET_COMPOSERS:
    comp_dir = os.path.join(FILTERED_DIR, comp)
    for f in os.listdir(comp_dir):
        try:
            pm = pretty_midi.PrettyMIDI(os.path.join(comp_dir, f))
            tempos.extend(pm.get_tempo_changes()[1])
        except:
            continue

plt.hist(tempos, bins=40)
plt.title("Tempo Distribution")
plt.xlabel("BPM")
plt.ylabel("Frequency")
plt.show()


##### 2.5 Extract Global Statistics - Sequence length distribution (LSTM)

In [0]:
seq_lengths = []

for comp in TARGET_COMPOSERS:
    comp_dir = os.path.join(FILTERED_DIR, comp)
    for f in os.listdir(comp_dir):
        try:
            pm = pretty_midi.PrettyMIDI(os.path.join(comp_dir, f))
            count = sum(len(inst.notes) for inst in pm.instruments)
            seq_lengths.append(count)
        except:
            continue

plt.hist(seq_lengths, bins=50)
plt.title("Sequence Length Distribution")
plt.xlabel("Number of Notes")
plt.ylabel("Frequency")
plt.show()


##### 2.6 Extract Global Statistics - Piano-roll density (for CNN)

In [0]:
densities = []

for comp in TARGET_COMPOSERS:
    comp_dir = os.path.join(FILTERED_DIR, comp)
    for f in os.listdir(comp_dir):
        try:
            pm = pretty_midi.PrettyMIDI(os.path.join(comp_dir, f))
            pr = pm.get_piano_roll(fs=10)
            densities.append(np.count_nonzero(pr) / pr.size)
        except:
            continue

plt.hist(densities, bins=40)
plt.title("Piano-Roll Density Distribution")
plt.xlabel("Active Ratio")
plt.ylabel("Frequency")
plt.show()

LTSM Model - Input: (max_notes × 3), bidirectional LTSM and fully connected classifier
CNN Model - Input: (1 × pitch_range × frames), 3 convolutional blocks, global average pooling, fully connected classifier.

#### 3. Label encoding and file collection

In [0]:
label_map = {
    "Bach": 0,
    "Beethoven": 1,
    "Chopin": 2,
    "Mozart": 3
}

def collect_files(base_dir):
    files = []
    labels = []
    for comp in TARGET_COMPOSERS:
        comp_dir = os.path.join(base_dir, comp)
        for f in glob.glob(os.path.join(comp_dir, "*.mid")) + glob.glob(os.path.join(comp_dir, "*.midi")):
            files.append(f)
            labels.append(label_map[comp])
    return files, labels

all_files, all_labels = collect_files(FILTERED_DIR)
print("Total files:", len(all_files))

#### 4. Data preprocessing & feature extraction
Two represenations are built
LSTM: sequence of note events
CNN: piano-roll matrix

##### 4.1 MIDI to event sequence for LSTM

In [0]:
def midi_to_event_sequence(midi_path, max_notes=500):
    try:
        pm = pretty_midi.PrettyMIDI(midi_path)
    except Exception:
        return None  # skip problematic files

    events = []
    for instrument in pm.instruments:
        if instrument.is_drum:
            continue
        for note in instrument.notes:
            pitch = note.pitch
            start = note.start
            end = note.end
            duration = end - start
            velocity = note.velocity
            events.append([pitch, duration, velocity])

    # sort by start time (approximate using note.start)
    # pretty_midi doesn't keep start in note object list order guaranteed
    # but we can sort by start time if needed:
    # events = sorted(events, key=lambda x: x_start)

    if len(events) == 0:
        return None

    # truncate or pad
    if len(events) > max_notes:
        events = events[:max_notes]
    else:
        # pad with zeros
        pad_len = max_notes - len(events)
        events.extend([[0, 0.0, 0]] * pad_len)

    return np.array(events, dtype=np.float32)


##### 4.2 MIDI to piano-roll for CNN

In [0]:
def midi_to_pianoroll(midi_path, fs=10, max_frames=500, pitch_range=(21, 108)):
    # pitch_range: typical piano keys (A0–C8)
    try:
        pm = pretty_midi.PrettyMIDI(midi_path)
    except Exception:
        return None

    pr = pm.get_piano_roll(fs=fs)  # shape: (128, T)
    if pr.shape[1] == 0:
        return None

    # crop pitch range
    low, high = pitch_range
    pr = pr[low:high+1, :]  # (num_pitches, T)

    # normalize velocities to [0,1]
    pr = pr / 127.0

    # crop or pad time dimension
    if pr.shape[1] > max_frames:
        pr = pr[:, :max_frames]
    else:
        pad_width = max_frames - pr.shape[1]
        pr = np.pad(pr, ((0, 0), (0, pad_width)), mode='constant')

    return pr.astype(np.float32)


##### 4.3 music21 feature extraction (notes, chords, tempo)
Can be integrated later as additional channels or metadata

In [0]:
def extract_music21_features(midi_path):
    try:
        score = converter.parse(midi_path)
    except Exception:
        return None

    notes = []
    chords = []
    tempos = []

    for el in score.flat:
        if el.isNote:
            notes.append(el.pitch.midi)
        elif el.isChord:
            chords.append([p.midi for p in el.pitches])
        elif el.classes and "MetronomeMark" in el.classes:
            tempos.append(el.number)

    return {
        "notes": notes,
        "chords": chords,
        "tempos": tempos
    }

##### 5. Model Building

In [0]:

all_files, all_labels = collect_files(FILTERED_DIR)

X = []
y = []

# Now iterate through the collection to process the binary MIDI data
for file_path, label in zip(all_files, all_labels):
    # Uses your midi_to_event_sequence function to handle binary data
    data = midi_to_event_sequence(file_path, max_notes=500)
    
    if data is not None:
        X.append(data)
        y.append(label)

# Convert lists to NumPy arrays for training
X = np.array(X)
y = np.array(y)

print(f"Processed {len(X)} files for Bach, Beethoven, Chopin, and Mozart.")

In [0]:
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, TimeDistributed, Embedding, RepeatVector, Concatenate, Reshape, Conv1D, MaxPooling1D, Flatten, BatchNormalization, GRU, Conv2D, MaxPooling2D, AveragePooling2D

# Define restraints
max_notes = 500
input_shape = (max_notes, 3)
num_classes = 4

# Build model
model = Sequential()

# Convolutional layer 1
model.add(Conv1D(64, kernel_size = 3, padding = 'same', activation='relu', input_shape=input_shape))
model.add(BatchNormalization())
model.add(MaxPooling1D(pool_size=2))

# Convolutional layer 2
model.add(Conv1D(128, kernel_size = 3, padding = 'same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling1D(pool_size = 2))

# Convolutional layer 3
model.add(Conv1D(256, kernel_size = 3, padding = 'same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling1D(pool_size = 2))

# LSTM layer
model.add(LSTM(128, return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(128))

# Dense layer
model.add(Dense(128, activation = 'relu'))
model.add(Dropout(0.4))

# Output layer
model.add(Dense(num_classes, activation = 'softmax'))

# Compile model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

#### 6. Model Training

In [0]:
# Split training, test, validation data
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Convert labels to one-hot encoding
y_train = to_categorical(y_train, num_classes = 4)
y_val = to_categorical(y_val, num_classes = 4)
y_test = to_categorical(y_test, num_classes = 4)

# Fit model
model.fit(X_train, y_train, epochs=15, batch_size=32, validation_data=(X_val, y_val))

In [0]:
# Confusion matrix
from sklearn.metrics import confusion_matrix

y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

cm = confusion_matrix(y_true_classes, y_pred_classes)
print(cm)